In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
dir1 = os.path.dirname(os.path.abspath(''))
if not dir1 in sys.path: sys.path.append(dir1)

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np
from tqdm.auto import trange

from utils.plots import *

from utils.signal import find_ecg_peaks

data_dir = Path('../../data/aurora/')
df_dir = Path('../dfs')
df_dir.mkdir(exist_ok=True, parents=True)


In [3]:
df_meta_aus = pd.read_csv(data_dir/'measurements_auscultatory.tsv', delimiter='\t')
df_meta_aus['bp_method'] = 'auscultatory'
df_meta_osc = pd.read_csv(data_dir/'measurements_oscillometric.tsv', delimiter='\t')
df_meta_osc['bp_method'] = 'oscillometric'
df_meta = pd.concat([df_meta_aus, df_meta_osc])
df_meta.dropna(subset=['waveform_file_path'], inplace=True)
df_meta.reset_index(drop=True, inplace=True)

df_feat = pd.read_csv(data_dir/'features.tsv', delimiter='\t')
df_feat.drop(columns=['date_time', 'sbp', 'dbp'], inplace=True)

df_part = pd.read_csv(data_dir/'participants.tsv', delimiter='\t')
df_part['bmi'] = df_part['weight'] / df_part['height']**2

df_all = pd.merge(df_meta, df_feat, how='outer', on=['pid', 'phase', 'measurement'])
df_all = pd.merge(df_all, df_part, how='outer', on=['pid'])
df_all.dropna(subset=['waveform_file_path'], inplace=True)
df_all.sort_values(['pid', 'date_time'], inplace=True)
df_all.reset_index(drop=True, inplace=True)


In [4]:
all_locs = [None] * len(df_meta)
for i in trange(len(df_meta)):
    fpath = data_dir / df_meta.iloc[i].waveform_file_path
    df_wav = pd.read_csv(fpath, sep='\t')
    
    t = df_wav.t
    ecg = df_wav.ekg
    delta_t = np.median(np.diff(t))
    assert np.all(np.isclose(np.diff(t), delta_t))
    fs = 1 / delta_t
    
    result = find_ecg_peaks(ecg, fs)
    if result is None:
        continue

    all_locs[i] = np.squeeze(result[0])


  0%|          | 0/38623 [00:00<?, ?it/s]

In [5]:
valid_ecg = [x is not None for x in all_locs]
df_all['valid_ecg'] = valid_ecg

df_all.to_csv(df_dir/'df_all.csv', index=False)
